## LQR GTSAM

In [1]:
import gtdynamics as gtd

## Mujoco Balancing Sim

### No policy, just start a simulation

In [2]:
import mujoco
import mujoco.viewer
import numpy
import gtdynamics as gtd
import gtsam

import mediapy as media

In [3]:
URDF_PATH = '../../models/urdfs/my_cart_pole.urdf'
MJCF_PATH = '../../models/mjcf/cart_pole.xml'

In [4]:
# MuJoCo Viewer Playground. Only run for the playground
model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

#mujoco.viewer.launch(model, data)

In [5]:
# Load cart-pole robot from URDF file
robot = gtd.CreateRobotFromFile(URDF_PATH)

robot = robot.fixLink("base_link")  # Fix the base link

# Link and Joints
link_names = [(link.id(), link.name()) for link in robot.links()]
link_names.sort()
print("Links:")
for id, name in link_names:
    print(f"  Link {id}: {name}")

joint_names = [(joint.id(), joint.name()) for joint in robot.joints()]
joint_names.sort()
print("\nJoints:")
for id, name in joint_names:
    joint = robot.joint(name)
    print(f"  Joint {id}: {name} (type: {joint.type()})")

print(f"Total links: {robot.numLinks()}")
print(f"Total joints: {robot.numJoints()}")

Links:
  Link 0: base_link
  Link 1: cart
  Link 2: pole

Joints:
  Joint 0: cart_slider (type: Type.Prismatic)
  Joint 1: pole_hinge (type: Type.Revolute)
Total links: 3
Total joints: 2


In [6]:
VIDEO_WIDTH = 800
VIDEO_HEIGHT = 600

In [7]:
model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

In [8]:
duration = 3.8  # (seconds)
framerate = 60  # (Hz)

# Simulate and display video.
frames = []

mujoco.mj_resetData(model, data)  
data.qpos[1] = 0.02  # Small initial petrubation of a pole angle

with mujoco.Renderer(model, width=VIDEO_WIDTH, height=VIDEO_HEIGHT) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * framerate:
      renderer.update_scene(data, camera="wide_view")
      pixels = renderer.render()
      frames.append(pixels)

media.show_video(frames, fps=framerate, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)

libdecor-gtk-WARNING: Failed to initialize GTK
Failed to load plugin 'libdecor-gtk.so': failed to init
No plugins found, falling back on no decorations


This browser does not support the video tag.


In [ ]:
# Parameters for balancing

balance_time = 3 # seconds 0.1 -- 0.25 -- 5   
balance_steps = 300  #        2   -- 5    -- 100
balance_dt = balance_time / balance_steps # 0.05 sec = 50ms

initial_pole_perturbation = 0.02  # Small angle deviation from upright (radians)
target_pole_angle = 0             # Straight up 

print(f"Balancing parameters:")
print(f"  Time horizon: {balance_time} seconds")
print(f"  Time steps: {balance_steps}")
print(f"  dt: {balance_dt:.4f} seconds")
print(f"  Initial perturbation: {initial_pole_perturbation:.3f} rad ({numpy.degrees(initial_pole_perturbation):.1f}°)")

# Set up noise models for balancing
simga_soft_objective = 1e-2
sigma_dynamics_balance = 1e-6     
sigma_objectives_balance = 1e-4   
sigma_torque_balance = 1e0        

# Create noise models
dynamics_model_6_bal = gtsam.noiseModel.Isotropic.Sigma(6, sigma_dynamics_balance)
dynamics_model_1_bal = gtsam.noiseModel.Isotropic.Sigma(1, sigma_dynamics_balance)
objectives_model_6_bal = gtsam.noiseModel.Isotropic.Sigma(6, sigma_objectives_balance)
objectives_model_1_bal = gtsam.noiseModel.Isotropic.Sigma(1, sigma_objectives_balance)
soft_objectives_model_1_bal = gtsam.noiseModel.Isotropic.Sigma(1, simga_soft_objective)
torque_model_bal = gtsam.noiseModel.Isotropic.Sigma(1, sigma_torque_balance)
constrained_model_1 = gtsam.noiseModel.Constrained.All(1)


Balancing parameters:
  Time horizon: 3 seconds
  Time steps: 300
  dt: 0.0100 seconds
  Initial perturbation: 0.020 rad (1.1°)


In [10]:
# Create FG
gravity_vec = numpy.array([0, 0, -9.8])
opt_balance = gtd.OptimizerSetting(sigma_dynamics_balance)
graph_builder_balance = gtd.DynamicsGraph(opt_balance, gravity_vec, None)

balance_graph = graph_builder_balance.trajectoryFG(robot, balance_steps, balance_dt)

print(f"Balancing FG created with {balance_graph.size()} factors")

Balancing FG created with 5715 factors


In [11]:
# Add priors to enforce zero torque on the pole joint (joint 1) at each time steps
# The pole joint is not actuated, so its torque should always be zero
num_pole_torque_priors = 0
for t in range(balance_steps + 1):
    balance_graph.addPriorDouble(gtd.TorqueKey(1, t), 0.0, constrained_model_1)
    num_pole_torque_priors += 1

print(f"Added {num_pole_torque_priors} zero torque priors for pole joint")


Added 301 zero torque priors for pole joint


In [12]:
# Add initial conditions. Pole's angle is peturbed - 0.02
initial_cart_pos_bal = 0.0
initial_pole_angle_bal = target_pole_angle + initial_pole_perturbation

# Add priors for initial joint angles and velocities
balance_graph.addPriorDouble(gtd.JointAngleKey(0, 0), initial_cart_pos_bal, dynamics_model_1_bal)
balance_graph.addPriorDouble(gtd.JointAngleKey(1, 0), initial_pole_angle_bal, dynamics_model_1_bal)
balance_graph.addPriorDouble(gtd.JointVelKey(0, 0), 0.0, dynamics_model_1_bal)
balance_graph.addPriorDouble(gtd.JointVelKey(1, 0), 0.0, dynamics_model_1_bal)


print(f"Initial conditions set:")
print(f"  Cart position: {initial_cart_pos_bal}")
print(f"  Pole angle: {initial_pole_angle_bal:.4f} rad ({numpy.degrees(initial_pole_angle_bal):.1f}°)")
print(f"  Initial velocities: 0.0")


Initial conditions set:
  Cart position: 0.0
  Pole angle: 0.0200 rad (1.1°)
  Initial velocities: 0.0


In [13]:
# Add pole angle objective at each step starting at 50
for t in range(1, 50):
    balance_graph.addPriorDouble(gtd.JointAngleKey(1, t), target_pole_angle, soft_objectives_model_1_bal)

for t in range(50, balance_steps + 1):
    balance_graph.addPriorDouble(gtd.JointAngleKey(1, t), target_pole_angle, objectives_model_1_bal)

In [14]:
# Final state at rest
final_time_bal = balance_steps

balance_graph.addPriorDouble(gtd.JointAngleKey(0, final_time_bal), initial_cart_pos_bal, dynamics_model_1_bal) # Position of a cart at the end
balance_graph.addPriorDouble(gtd.JointVelKey(0, final_time_bal), 0.0, objectives_model_1_bal)  # Cart at rest
balance_graph.addPriorDouble(gtd.JointVelKey(1, final_time_bal), 0.0, objectives_model_1_bal)  # Pole at rest
balance_graph.addPriorDouble(gtd.JointAccelKey(0, final_time_bal), 0.0, objectives_model_1_bal)  # Cart at rest
balance_graph.addPriorDouble(gtd.JointAccelKey(1, final_time_bal), 0.0, objectives_model_1_bal)  # Pole at rest

In [15]:
# Torque minimization 
for t in range(balance_steps + 1):
    torque_key = gtd.TorqueKey(0, t)  # Cart torque
    min_torque_factor = gtd.MinTorqueFactor(torque_key, torque_model_bal)
    balance_graph.add(min_torque_factor)

In [16]:
# Create initial guess - all zeros
initializer_bal = gtd.Initializer()
initial_values_bal = initializer_bal.ZeroValuesTrajectory(robot, balance_steps, 0, 0.0, None)

In [17]:
initial_values_bal.size()

6321

In [18]:
print(f"FG final size: {balance_graph.size()} factors")

FG final size: 6626 factors


In [19]:
# Set up optimizer for balancing
print("Setting up optimizer for balancing...")
params_bal = gtsam.LevenbergMarquardtParams()
params_bal.setVerbosityLM("SUMMARY")
params_bal.setMaxIterations(100)
params_bal.setRelativeErrorTol(1e-8)
params_bal.setAbsoluteErrorTol(1e-8)

# Calculate initial error
#initial_error_bal = balance_graph.error(initial_values_bal)
#print(f"Problem size: {balance_graph.size()} factors, {initial_values_bal.size()} variables")
#print(f"Initial error: {initial_error_bal:.6e}")

# Create and run optimizer
optimizer_bal = gtsam.LevenbergMarquardtOptimizer(balance_graph, initial_values_bal, params_bal)

Setting up optimizer for balancing...


In [20]:
gtd.GTDKeyFormatter(18296968702853120)

'A[0]0'

In [21]:
result_bal = optimizer_bal.optimize()

Initial error: 1.45986e+16, values: 6321
iter      cost      cost_change    lambda  success iter_time
   0  8.99694e+10      1.5e+16      1e-05      1       0.07
   1           56        9e+10      1e-06      1       0.09
   2           26           30      1e-07      1       0.05
   3           26      7.5e-14      1e-08      1       0.04


In [22]:
# Extract and plot data
print("Extracting balancing trajectory...")
balance_times = []
balance_cart_positions = []
balance_cart_velocities = []
balance_cart_torques = []
balance_pole_angles = []
balance_pole_velocities = []

for t in range(balance_steps):
    balance_times.append(t * balance_dt)
    
    try:
        cart_pos = gtd.JointAngle(result_bal, 0, t)
        cart_vel = gtd.JointVel(result_bal, 0, t)
        cart_torque = gtd.Torque(result_bal, 0, t)
        pole_angle = gtd.JointAngle(result_bal, 1, t)
        pole_vel = gtd.JointVel(result_bal, 1, t)
        
        balance_cart_positions.append(cart_pos)
        balance_cart_velocities.append(cart_vel)
        balance_cart_torques.append(cart_torque)
        balance_pole_angles.append(pole_angle)
        balance_pole_velocities.append(pole_vel)
    except:
        print("Couldn't retrieve some values")
        # Fallback values
        balance_cart_positions.append(0.0)
        balance_cart_velocities.append(0.0)
        balance_cart_torques.append(0.0)
        balance_pole_angles.append(target_pole_angle)
        balance_pole_velocities.append(0.0)

# Convert to numpy arrays
balance_times = numpy.array(balance_times)
balance_cart_positions = numpy.array(balance_cart_positions)
balance_cart_velocities = numpy.array(balance_cart_velocities)
balance_cart_torques = numpy.array(balance_cart_torques)
balance_pole_angles = numpy.array(balance_pole_angles)
balance_pole_velocities = numpy.array(balance_pole_velocities)

# Create interactive visualization with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Pole Balancing Performance',
        'Cart Motion for Balancing',
        'System Velocities',
        'Control Torque'
    ),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Plot 1: Pole angle deviation from upright
pole_deviations = numpy.degrees(balance_pole_angles - target_pole_angle)
fig.add_trace(
    go.Scatter(x=balance_times, y=pole_deviations, name='Pole Deviation',
               line=dict(color='blue', width=2), showlegend=True),
    row=1, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5,
              annotation_text="Target (upright)", row=1, col=1)

# Plot 2: Cart position
fig.add_trace(
    go.Scatter(x=balance_times, y=balance_cart_positions, name='Cart Position',
               line=dict(color='green', width=2), showlegend=False),
    row=1, col=2
)

# Plot 3: Velocities
fig.add_trace(
    go.Scatter(x=balance_times, y=balance_cart_velocities, name='Cart Velocity',
               line=dict(color='green', width=2)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=balance_times, y=numpy.degrees(balance_pole_velocities), name='Pole Velocity (°/s)',
               line=dict(color='blue', width=2)),
    row=2, col=1
)

# Plot 4: Control torque
fig.add_trace(
    go.Scatter(x=balance_times, y=balance_cart_torques, name='Torque',
               line=dict(color='purple', width=2), showlegend=False),
    row=2, col=2
)

# Update layout
fig.update_layout(
    title="Pole Balancing Trajectory Analysis",
    height=800,
    showlegend=True
)

# Update axis labels
fig.update_xaxes(title_text="Time (s)", row=1, col=1)
fig.update_xaxes(title_text="Time (s)", row=1, col=2)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_xaxes(title_text="Time (s)", row=2, col=2)

fig.update_yaxes(title_text="Pole Angle Deviation (°)", row=1, col=1)
fig.update_yaxes(title_text="Cart Position (m)", row=1, col=2)
fig.update_yaxes(title_text="Velocities", row=2, col=1)
fig.update_yaxes(title_text="Torque (N⋅m)", row=2, col=2)

print("Plotting complete!")
fig.show()

Extracting balancing trajectory...
Plotting complete!


In [23]:
# Analyze factor contributions to final error
print("Analyzing factor contributions to final error...")

# Get all factors and their errors
factor_errors = []
factor_types = []

for i in range(balance_graph.size()):
    factor = balance_graph.at(i)
    error = factor.error(result_bal)
    factor_errors.append(error)
    
    # Get the actual factor type name using Python's type system
    factor_type = type(factor).__name__
    factor_types.append(factor_type)

# Convert to numpy arrays
factor_errors = numpy.array(factor_errors)
factor_types = numpy.array(factor_types)

# Group by factor type
unique_types = numpy.unique(factor_types)
type_errors = {}
type_counts = {}

for ftype in unique_types:
    mask = factor_types == ftype
    type_errors[ftype] = numpy.sum(factor_errors[mask])
    type_counts[ftype] = numpy.sum(mask)

# Sort by error contribution
sorted_types = sorted(type_errors.keys(), key=lambda x: type_errors[x], reverse=True)

print(f"\nFactor Error Analysis:")
print(f"Total factors: {len(factor_errors)}")
print(f"Total error: {numpy.sum(factor_errors):.6e}")
print(f"\nError by factor type:")
for ftype in sorted_types:
    print(f"  {ftype:20s}: {type_errors[ftype]:12.6e} ({type_counts[ftype]:4d} factors, avg: {type_errors[ftype]/type_counts[ftype]:.6e})")

# Create visualization with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Error by Factor Type',
        'Number of Factors by Type',
        'Average Error per Factor by Type',
        'Error Distribution (Log Scale)'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "histogram"}]]
)

# Plot 1: Total error by factor type
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Total Error'),
    row=1, col=1
)

# Plot 2: Number of factors by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Count'),
    row=1, col=2
)

# Plot 3: Average error per factor
avg_errors = [type_errors[t]/type_counts[t] for t in sorted_types]
fig.add_trace(
    go.Bar(x=sorted_types, y=avg_errors,
           marker_color=colors[:len(sorted_types)],
           name='Avg Error'),
    row=2, col=1
)

# Plot 4: Error distribution histogram (log scale)
fig.add_trace(
    go.Histogram(x=numpy.log10(factor_errors[factor_errors > 0] + 1e-20),
                 nbinsx=50,
                 marker_color='steelblue',
                 name='Error Distribution'),
    row=2, col=2
)

# Update layout
fig.update_layout(
    title=f"Factor Error Contribution Analysis (Total Error: {numpy.sum(factor_errors):.6e})",
    height=800,
    showlegend=False
)

# Update axis labels
fig.update_xaxes(title_text="Factor Type", row=1, col=1)
fig.update_xaxes(title_text="Factor Type", row=1, col=2)
fig.update_xaxes(title_text="Factor Type", row=2, col=1)
fig.update_xaxes(title_text="log10(Error)", row=2, col=2)

fig.update_yaxes(title_text="Total Error", type="log", row=1, col=1)
fig.update_yaxes(title_text="Number of Factors", row=1, col=2)
fig.update_yaxes(title_text="Average Error", type="log", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=2)

# Rotate x-axis labels for better readability
fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_xaxes(tickangle=-45, row=1, col=2)
fig.update_xaxes(tickangle=-45, row=2, col=1)

print("\n📊 Visualization created! Check the plots above.")
fig.show()

# Additional analysis: Top 10 individual factors with highest error
print("\nTop 10 individual factors with highest error:")
top_indices = numpy.argsort(factor_errors)[-10:][::-1]
for rank, idx in enumerate(top_indices, 1):
    factor = balance_graph.at(idx)
    print(f"  {rank:2d}. Factor {idx:4d} ({factor_types[idx]:15s}): error = {factor_errors[idx]:.6e}")


Analyzing factor contributions to final error...

Factor Error Analysis:
Total factors: 6626
Total error: 2.617905e+01

Error by factor type:
  NonlinearFactor     : 1.778929e+01 (5414 factors, avg: 3.285794e-03)
  PriorFactorDouble   : 6.142326e+00 ( 610 factors, avg: 1.006939e-02)
  MinTorqueFactor     : 2.247433e+00 ( 301 factors, avg: 7.466556e-03)
  PriorFactorPose3    : 2.408670e-09 ( 301 factors, avg: 8.002226e-12)

📊 Visualization created! Check the plots above.



Top 10 individual factors with highest error:
   1. Factor   17 (NonlinearFactor): error = 3.294570e+00
   2. Factor   36 (NonlinearFactor): error = 2.633238e+00
   3. Factor   55 (NonlinearFactor): error = 2.108685e+00
   4. Factor   74 (NonlinearFactor): error = 1.692037e+00
   5. Factor 6020 (PriorFactorDouble): error = 1.523408e+00
   6. Factor   93 (NonlinearFactor): error = 1.360603e+00
   7. Factor 6021 (PriorFactorDouble): error = 1.158159e+00
   8. Factor  112 (NonlinearFactor): error = 1.096538e+00
   9. Factor  131 (NonlinearFactor): error = 8.857930e-01
  10. Factor 6022 (PriorFactorDouble): error = 8.781719e-01


In [24]:
# Reset MuJoCo simulation to clean state
mujoco.mj_resetData(model, data)

# Set initial conditions to match our optimization
# Initial cart position = 0, pole slightly perturbed (0.02 rad)
data.qpos[0] = initial_cart_pos_bal  # Cart position (should be 0.0)
data.qpos[1] = initial_pole_angle_bal  # Pole angle (0.02 rad perturbation)
data.qvel[0] = 0.0  # Cart velocity
data.qvel[1] = 0.0  # Pole angular velocity

print(f"Initial MuJoCo state set:")
print(f"  Cart position: {data.qpos[0]:.4f} m")
print(f"  Pole angle: {data.qpos[1]:.4f} rad ({numpy.degrees(data.qpos[1]):.1f}°)")
print(f"  Velocities: [{data.qvel[0]:.3f}, {data.qvel[1]:.3f}]")

# Simulation parameters - make physics timestep equal to factor-graph dt
control_dt = balance_dt      # control timestep (same as factor graph)
physics_dt = balance_dt      # MuJoCo physical timestep set equal to factor-graph dt

render_freq = 60  # Render at 60 FPS for smooth video
frames_per_control = max(1, int(render_freq * control_dt))  # How many video frames per control step

# Calculate total expected frames
total_expected_frames = int(balance_time * render_freq)  # e.g. balance_time * 60

# Set MuJoCo timestep
model.opt.timestep = physics_dt  # Set physics timestep equal to factor-graph dt

# Storage for MuJoCo simulation results
mujoco_times = []
mujoco_cart_positions = []
mujoco_cart_velocities = []
mujoco_pole_angles = []
mujoco_pole_velocities = []
mujoco_applied_torques = []

# Video recording setup
frames = []
video_width = VIDEO_WIDTH
video_height = VIDEO_HEIGHT

print(f"Starting MuJoCo simulation with optimized torques...")
print("Recording video for playback...")

with mujoco.Renderer(model, width=video_width, height=video_height) as renderer:
    for step in range(balance_steps):
        current_time = step * control_dt
        
        # Get the optimized torque for this time step
        if step < len(balance_cart_torques):
            target_torque = balance_cart_torques[step]
        else:
            target_torque = 0.0  # Fallback
        
        # Apply torque to cart (actuator 0 controls joint 0 - cart)
        data.ctrl[0] = target_torque * 100
        
        # Step simulation once per control timestep (physics_dt == control_dt)
        mujoco.mj_step(model, data)
        
        # Record current state
        mujoco_times.append(current_time)
        mujoco_cart_positions.append(data.qpos[0])  # Cart position
        mujoco_cart_velocities.append(data.qvel[0])  # Cart velocity  
        mujoco_pole_angles.append(data.qpos[1])     # Pole angle
        mujoco_pole_velocities.append(data.qvel[1]) # Pole angular velocity
        mujoco_applied_torques.append(target_torque)

        # Render multiple frames per control step for smooth 60 FPS video
        for frame_idx in range(frames_per_control):
            renderer.update_scene(data, camera="wide_view")
            pixels = renderer.render()
            frames.append(pixels)

# Convert results to numpy arrays
mujoco_times = numpy.array(mujoco_times)
mujoco_cart_positions = numpy.array(mujoco_cart_positions)
mujoco_cart_velocities = numpy.array(mujoco_cart_velocities)
mujoco_pole_angles = numpy.array(mujoco_pole_angles)
mujoco_pole_velocities = numpy.array(mujoco_pole_velocities)
mujoco_applied_torques = numpy.array(mujoco_applied_torques)

print(mujoco_cart_positions[-1])

media.show_video(frames, fps=render_freq, width=video_width, height=video_height)


Initial MuJoCo state set:
  Cart position: 0.0000 m
  Pole angle: 0.0200 rad (1.1°)
  Velocities: [0.000, 0.000]
Starting MuJoCo simulation with optimized torques...
Recording video for playback...
30.4687494936068


This browser does not support the video tag.


### LQR Balancing

In [25]:
# After optimizer_bal.optimize()
print("Linearizing the graph around the optimal trajectory...")
gaussian_graph_bal = balance_graph.linearize(result_bal)
print("Linearization complete.")

Linearizing the graph around the optimal trajectory...
Linearization complete.


In [ ]:
gaussian_graph_bal.dot()

In [ ]:
gtd.GTDKeyFormatter(19705451687968817)

In [26]:
print("Creating backward elimination ordering including ALL keys...")
ordering_bal = gtsam.Ordering()

num_links = robot.numLinks()
num_joints = robot.numJoints()

# Add variables from t = balance_steps down to t = 0 
for t in reversed(range(balance_steps + 1)):
    # Torques (controls)
    for j in range(num_joints): 
         key = gtd.TorqueKey(j, t)
         if result_bal.exists(key): ordering_bal.push_back(key)

    # Poses/Angles (state)
    for j in range(num_joints):
        key = gtd.JointAngleKey(j, t)
        if result_bal.exists(key): ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.PoseKey(i, t)
        if result_bal.exists(key): ordering_bal.push_back(key)

    # Twists (state)
    for j in range(num_joints):
        key = gtd.JointAccelKey(j, t)
        if result_bal.exists(key): ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.TwistAccelKey(i, t)
        if result_bal.exists(key): ordering_bal.push_back(key)

    # Wrenches (state)
    for i in range(num_links):
        for j in range(num_joints):
            key = gtd.WrenchKey(i, j, t)
            if result_bal.exists(key): ordering_bal.push_back(key)

    # Velocities (state)
    for j in range(num_joints):
        key = gtd.JointVelKey(j, t)
        if result_bal.exists(key): ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.TwistKey(i, t)
        if result_bal.exists(key): ordering_bal.push_back(key)


print(f"Elimination ordering created with {ordering_bal.size()} keys.")

# Check if ordering size matches result size
if ordering_bal.size() != result_bal.size():
    print(f"Warning: Ordering size ({ordering_bal.size()}) does not match result size ({result_bal.size()}). Keys might be missing or duplicated!")
    # Debugging
    result_keys_set = set(result_bal.keys())
    ordering_keys_list = [ordering_bal.at(i) for i in range(ordering_bal.size())]
    ordering_keys_set = set(ordering_keys_list)
    missing_keys = result_keys_set - ordering_keys_set
    extra_keys = ordering_keys_set - result_keys_set
    if missing_keys:
        print("Keys in result but not in ordering:")
        # for key in missing_keys: print(gtsam.DefaultKeyFormatter(key))
        print(f"(Total missing: {len(missing_keys)})")
        print(missing_keys)
    if extra_keys:
        print("Keys in ordering but not in result (problematic):")
        # for key in extra_keys: print(gtsam.DefaultKeyFormatter(key))
        print(f"(Total extra: {len(extra_keys)})")
    if len(ordering_keys_list) != len(ordering_keys_set):
        print("Warning: Duplicate keys found in ordering!")
else:
     print("Ordering size matches result size. Elimination should work.")

Creating backward elimination ordering including ALL keys...
Elimination ordering created with 6321 keys.
Ordering size matches result size. Elimination should work.


In [27]:
# Eliminate to get the Bayes Net (encodes the policy)
policy_bayes_net_bal = gaussian_graph_bal.eliminateSequential(ordering_bal)

RuntimeError: 
Indeterminant linear system detected while working near variable
19705451687968816 (Symbol: 19705451687968816).

Thrown when a linear system is ill-posed.  The most common cause for this
error is having underconstrained variables.  Mathematically, the system is
underdetermined.  See the GTSAM Doxygen documentation at
http://borg.cc.gatech.edu/ on gtsam::IndeterminantLinearSystemException for
more information.

In [ ]:
import graphviz

In [ ]:
graphviz.Source(policy_bayes_net_bal.dot())

In [ ]:
help(policy_bayes_net_bal)

In [ ]:
# Reset MuJoCo simulation to clean state
mujoco.mj_resetData(model, data)

# Set initial conditions to match optimization
data.qpos[0] = initial_cart_pos_bal  # 0.0
data.qpos[1] = initial_pole_angle_bal  # 0.02
data.qvel[0] = 0.0
data.qvel[1] = 0.0

# Make MuJoCo physics timestep explicitly match factor-graph dt
control_dt = balance_dt
physics_dt = balance_dt
model.opt.timestep = physics_dt  # Use same physics timestep as optimization

print("Starting Mujoco simulation with TVLQR FEEDBACK policy...")
frames = []

# Data storage for tracking deviations and control
tracking_data = {
    'time': [],
    'cart_pos_deviation': [],
    'pole_angle_deviation': [],
    'cart_vel_deviation': [],
    'pole_vel_deviation': [],
    'torque_correction': [],
    'torque_reference': [],
    'torque_total': []
}

# Rendering / timing parameters
render_freq = 60  # fps
frames_per_control_step = max(1, int(control_dt * render_freq))

with mujoco.Renderer(model, width=VIDEO_WIDTH, height=VIDEO_HEIGHT) as renderer:
    
    for t in range(balance_steps):  # Loop from t=0 to t=N-1
        current_time = t * control_dt
        
        # Actual state
        q_actual = data.qpos
        v_actual = data.qvel
        
        # References from result_bal
        q_ref = numpy.array([
            gtd.JointAngle(result_bal, 0, t),
            gtd.JointAngle(result_bal, 1, t)
        ])
        v_ref = numpy.array([
            gtd.JointVel(result_bal, 0, t),
            gtd.JointVel(result_bal, 1, t)
        ])
        u_ref = gtd.Torque(result_bal, 0, t)

        # Find deviations dx
        dx = gtsam.VectorValues()
        dx.insert(gtd.JointAngleKey(0, t), numpy.array([q_actual[0] - q_ref[0]]))
        dx.insert(gtd.JointAngleKey(1, t), numpy.array([q_actual[1] - q_ref[1]]))
        dx.insert(gtd.JointVelKey(0, t), numpy.array([v_actual[0] - v_ref[0]]))
        dx.insert(gtd.JointVelKey(1, t), numpy.array([v_actual[1] - v_ref[1]]))

        # Filter the Bayes net to remove state variables at current time
        provided_keys = set([
            gtd.JointAngleKey(0, t),
            gtd.JointAngleKey(1, t),
            gtd.JointVelKey(0, t),
            gtd.JointVelKey(1, t)
        ])
        
        # Create a new Bayes net without the provided state variables
        filtered_bayes_net = gtsam.GaussianBayesNet()
        for i in range(policy_bayes_net_bal.size()):
            conditional = policy_bayes_net_bal.at(i)
            frontal_key = conditional.firstFrontalKey()
            if frontal_key not in provided_keys:
                filtered_bayes_net.push_back(conditional)
        
        # Use filtered bayes net to find corrections
        optimal_corrections = filtered_bayes_net.optimize(dx)
        
        # Extract the torque correction for the cart (joint 0)
        du = optimal_corrections.at(gtd.TorqueKey(0, t))[0]

        # Apply du
        u_total = u_ref + du
        data.ctrl[0] = u_total
        
        # Store tracking data
        tracking_data['time'].append(current_time)
        tracking_data['cart_pos_deviation'].append(q_actual[0] - q_ref[0])
        tracking_data['pole_angle_deviation'].append(q_actual[1] - q_ref[1])
        tracking_data['cart_vel_deviation'].append(v_actual[0] - v_ref[0])
        tracking_data['pole_vel_deviation'].append(v_actual[1] - v_ref[1])
        tracking_data['torque_correction'].append(du)
        tracking_data['torque_reference'].append(u_ref)
        tracking_data['torque_total'].append(u_total)

        # Simulate and render frames for this control timestep
        # Each control update (control_dt) should have multiple rendered frames
        substeps_per_frame = max(1, int(control_dt / (frames_per_control_step * model.opt.timestep)))
        
        for frame_idx in range(frames_per_control_step):
            # Run MuJoCo substeps between frames
            for _ in range(substeps_per_frame):
                mujoco.mj_step(model, data)
            
            # Render frame
            renderer.update_scene(data, camera="wide_view")
            pixels = renderer.render()
            frames.append(pixels)

print("Simulation with TVLQR feedback complete.")

# Show the video
media.show_video(frames, fps=render_freq, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)


In [ ]:
import matplotlib.pyplot as plt

# Create figure with subplots
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('TVLQR Feedback Control - Deviations and Corrections', fontsize=16)

time = tracking_data['time']

# Plot 1: Cart position deviation
axes[0, 0].plot(time, tracking_data['cart_pos_deviation'], 'b-', linewidth=2)
axes[0, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0, 0].set_ylabel('Deviation (m)', fontsize=11)
axes[0, 0].set_title('Cart Position Deviation', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Pole angle deviation
axes[0, 1].plot(time, numpy.rad2deg(tracking_data['pole_angle_deviation']), 'r-', linewidth=2)
axes[0, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0, 1].set_ylabel('Deviation (deg)', fontsize=11)
axes[0, 1].set_title('Pole Angle Deviation', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Cart velocity deviation
axes[1, 0].plot(time, tracking_data['cart_vel_deviation'], 'b-', linewidth=2)
axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1, 0].set_ylabel('Deviation (m/s)', fontsize=11)
axes[1, 0].set_title('Cart Velocity Deviation', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Pole angular velocity deviation
axes[1, 1].plot(time, numpy.rad2deg(tracking_data['pole_vel_deviation']), 'r-', linewidth=2)
axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1, 1].set_ylabel('Deviation (deg/s)', fontsize=11)
axes[1, 1].set_title('Pole Angular Velocity Deviation', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)

# Plot 5: Torque breakdown
axes[2, 0].plot(time, tracking_data['torque_reference'], 'g--', linewidth=2, label='Reference', alpha=0.7)
axes[2, 0].plot(time, tracking_data['torque_correction'], 'm-', linewidth=2, label='Correction (du)')
axes[2, 0].plot(time, tracking_data['torque_total'], 'k-', linewidth=2, label='Total Applied')
axes[2, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[2, 0].set_xlabel('Time (s)', fontsize=11)
axes[2, 0].set_ylabel('Torque (N·m)', fontsize=11)
axes[2, 0].set_title('Torque Components', fontsize=12)
axes[2, 0].legend(fontsize=10)
axes[2, 0].grid(True, alpha=0.3)

# Plot 6: Torque correction magnitude
axes[2, 1].plot(time, numpy.abs(tracking_data['torque_correction']), 'c-', linewidth=2)
axes[2, 1].set_xlabel('Time (s)', fontsize=11)
axes[2, 1].set_ylabel('|Correction| (N·m)', fontsize=11)
axes[2, 1].set_title('Torque Correction Magnitude', fontsize=12)
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Feedback Control Statistics ===")
print(f"Max cart position deviation: {max(numpy.abs(tracking_data['cart_pos_deviation'])):.6f} m")
print(f"Max pole angle deviation: {numpy.rad2deg(max(numpy.abs(tracking_data['pole_angle_deviation']))):.4f} deg")
print(f"Max torque correction: {max(numpy.abs(tracking_data['torque_correction'])):.4f} N·m")
print(f"Mean torque correction: {numpy.mean(numpy.abs(tracking_data['torque_correction'])):.4f} N·m")

In [ ]:
help(gtsam.GaussianConditional)